# Historical Data Analysis with Causal Inference

This notebook demonstrates how to use causal inference techniques on historical insurance data to understand the true effects of pricing changes.

## Objective
- Analyze historical pricing data
- Apply causal inference methods
- Estimate treatment effects
- Validate causal assumptions

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy import stats
from utils import generate_synthetic_insurance_data, calculate_treatment_effects, set_style

# Set plotting style
set_style()

print("Libraries imported successfully!")

## 1. Generate Historical-Style Data

Create a dataset that mimics historical insurance pricing data with temporal elements.

In [ ]:
# Generate historical data with time component
np.random.seed(42)

# Create data for different time periods
historical_data = []
base_date = pd.Timestamp('2020-01-01')

for period in range(24):  # 24 months of data
    # Generate data for each period
    period_data = generate_synthetic_insurance_data(n_samples=2000)
    
    # Add temporal elements
    period_data['period'] = period
    period_data['date'] = base_date + pd.DateOffset(months=period)
    
    # Add seasonal trends
    seasonal_factor = 1 + 0.1 * np.sin(2 * np.pi * period / 12)
    period_data['price'] *= seasonal_factor
    
    # Add market trend
    trend_factor = 1 + 0.02 * period  # 2% annual growth
    period_data['price'] *= trend_factor
    
    # Simulate price experiments in some periods
    if period in [6, 12, 18]:  # Price experiments
        experiment_customers = np.random.choice(period_data.index, size=500, replace=False)
        period_data.loc[experiment_customers, 'price'] *= 1.2  # 20% price increase
        period_data.loc[experiment_customers, 'price_experiment'] = 1
    else:
        period_data['price_experiment'] = 0
    
    historical_data.append(period_data)

# Combine all periods
df_historical = pd.concat(historical_data, ignore_index=True)

print(f"Historical dataset shape: {df_historical.shape}")
print(f"Time periods: {df_historical['period'].min()} to {df_historical['period'].max()}")
print(f"Date range: {df_historical['date'].min()} to {df_historical['date'].max()}")
print(f"Price experiments: {df_historical['price_experiment'].sum()} customers")

# Show sample data
print("\nSample historical data:")
print(df_historical[['period', 'date', 'price', 'price_experiment', 'conversion', 'profit']].head(10))

## 2. Exploratory Data Analysis

Analyze the historical data to understand trends and patterns.

In [ ]:
# Aggregate data by period
period_summary = df_historical.groupby('period').agg({
    'price': 'mean',
    'conversion': 'mean',
    'profit': 'mean',
    'price_experiment': 'sum',
    'customer_id': 'count'
}).rename(columns={'customer_id': 'customers'})

print("HISTORICAL DATA SUMMARY BY PERIOD")
print("=" * 50)
print(period_summary.head(12))

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Price trend over time
axes[0, 0].plot(period_summary.index, period_summary['price'], 'b-', linewidth=2, marker='o')
# Highlight experiment periods
experiment_periods = period_summary[period_summary['price_experiment'] > 0].index
axes[0, 0].scatter(experiment_periods, period_summary.loc[experiment_periods, 'price'], 
                  color='red', s=100, zorder=5, label='Price Experiments')
axes[0, 0].set_xlabel('Period')
axes[0, 0].set_ylabel('Average Price ($)')
axes[0, 0].set_title('Historical Price Trend')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Conversion rate over time
axes[0, 1].plot(period_summary.index, period_summary['conversion'], 'g-', linewidth=2, marker='s')
axes[0, 1].scatter(experiment_periods, period_summary.loc[experiment_periods, 'conversion'], 
                  color='red', s=100, zorder=5, label='Price Experiments')
axes[0, 1].set_xlabel('Period')
axes[0, 1].set_ylabel('Conversion Rate')
axes[0, 1].set_title('Historical Conversion Rate')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Profit over time
axes[1, 0].plot(period_summary.index, period_summary['profit'], 'purple', linewidth=2, marker='^')
axes[1, 0].scatter(experiment_periods, period_summary.loc[experiment_periods, 'profit'], 
                  color='red', s=100, zorder=5, label='Price Experiments')
axes[1, 0].set_xlabel('Period')
axes[1, 0].set_ylabel('Average Profit ($)')
axes[1, 0].set_title('Historical Profit per Customer')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Price vs Conversion relationship
axes[1, 1].scatter(period_summary['price'], period_summary['conversion'], 
                  c=period_summary.index, cmap='viridis', s=60)
axes[1, 1].set_xlabel('Average Price ($)')
axes[1, 1].set_ylabel('Conversion Rate')
axes[1, 1].set_title('Price vs Conversion (Time-Colored)')
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label('Time Period')

plt.tight_layout()
plt.show()

# Statistical summary
print("\nKEY STATISTICS:")
print(f"Average price growth per period: {(period_summary['price'].iloc[-1] / period_summary['price'].iloc[0])**(1/23) - 1:.2%}")
print(f"Price volatility (std): ${period_summary['price'].std():.2f}")
print(f"Conversion rate trend: {stats.linregress(period_summary.index, period_summary['conversion']).slope:.4f} per period")
print(f"Profit trend: ${stats.linregress(period_summary.index, period_summary['profit']).slope:.2f} per period")

## 3. Causal Inference Analysis

Apply causal inference methods to estimate the true effect of pricing changes.

In [ ]:
# Define treatment and control groups based on price experiments
treatment_group = df_historical[df_historical['price_experiment'] == 1]
control_group = df_historical[df_historical['price_experiment'] == 0]

print("CAUSAL INFERENCE ANALYSIS")
print("=" * 50)
print(f"Treatment group size: {len(treatment_group):,} customers")
print(f"Control group size: {len(control_group):,} customers")
print(f"Treatment periods: {sorted(treatment_group['period'].unique())}")

# 1. Simple difference in means (potentially biased)
simple_diff = treatment_group['conversion'].mean() - control_group['conversion'].mean()
t_stat, p_value = stats.ttest_ind(treatment_group['conversion'], control_group['conversion'])

print(f"\n1. SIMPLE DIFFERENCE IN MEANS:")
print(f"   Treatment conversion rate: {treatment_group['conversion'].mean():.4f}")
print(f"   Control conversion rate: {control_group['conversion'].mean():.4f}")
print(f"   Difference: {simple_diff:.4f}")
print(f"   T-statistic: {t_stat:.4f}")
print(f"   P-value: {p_value:.4f}")

# 2. Regression-based causal inference with controls
confounders = ['age', 'income', 'risk_score', 'previous_claims', 'period']
X = df_historical[confounders + ['price_experiment']]
y = df_historical['conversion']

# Fit regression model
causal_model = LinearRegression().fit(X, y)
treatment_effect = causal_model.coef_[-1]  # Coefficient for price_experiment

print(f"\n2. REGRESSION-BASED CAUSAL INFERENCE:")
print(f"   Treatment effect (controlled): {treatment_effect:.4f}")
print(f"   Model R-squared: {causal_model.score(X, y):.4f}")

# Show all coefficients
coef_df = pd.DataFrame({
    'variable': confounders + ['price_experiment'],
    'coefficient': causal_model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(f"\n   Model coefficients:")
for _, row in coef_df.iterrows():
    print(f"   {row['variable']:20}: {row['coefficient']:8.4f}")

# 3. Propensity score matching (simplified)
# Estimate propensity scores
propensity_model = LinearRegression().fit(X[confounders], df_historical['price_experiment'])
propensity_scores = propensity_model.predict(X[confounders])
df_historical['propensity_score'] = propensity_scores

print(f"\n3. PROPENSITY SCORE ANALYSIS:")
print(f"   Propensity score range: {propensity_scores.min():.4f} to {propensity_scores.max():.4f}")
print(f"   Treatment group avg propensity: {df_historical[df_historical['price_experiment']==1]['propensity_score'].mean():.4f}")
print(f"   Control group avg propensity: {df_historical[df_historical['price_experiment']==0]['propensity_score'].mean():.4f}")

# Visualize propensity scores
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df_historical[df_historical['price_experiment']==0]['propensity_score'], 
         bins=30, alpha=0.7, label='Control', color='blue')
plt.hist(df_historical[df_historical['price_experiment']==1]['propensity_score'], 
         bins=30, alpha=0.7, label='Treatment', color='red')
plt.xlabel('Propensity Score')
plt.ylabel('Frequency')
plt.title('Propensity Score Distribution')
plt.legend()

plt.subplot(1, 2, 2)
# Treatment effect by propensity score quartile
df_historical['prop_quartile'] = pd.qcut(df_historical['propensity_score'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
quartile_effects = []
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    q_data = df_historical[df_historical['prop_quartile'] == q]
    q_treatment = q_data[q_data['price_experiment'] == 1]['conversion'].mean()
    q_control = q_data[q_data['price_experiment'] == 0]['conversion'].mean()
    quartile_effects.append(q_treatment - q_control)

plt.bar(['Q1', 'Q2', 'Q3', 'Q4'], quartile_effects, color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.xlabel('Propensity Score Quartile')
plt.ylabel('Treatment Effect')
plt.title('Treatment Effect by Propensity Quartile')
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\n   Treatment effects by propensity quartile:")
for i, effect in enumerate(quartile_effects):
    print(f"   Q{i+1}: {effect:.4f}")

## 4. Robustness Checks

Validate our causal inference results with various robustness checks.

In [ ]:
print("ROBUSTNESS CHECKS")
print("=" * 50)

# 1. Placebo tests - check periods without experiments
non_experiment_periods = df_historical[~df_historical['period'].isin([6, 12, 18])]
placebo_results = []

for period in [3, 9, 15, 21]:  # Non-experiment periods
    period_data = non_experiment_periods[non_experiment_periods['period'] == period]
    if len(period_data) > 100:
        # Randomly assign "treatment" to 25% of customers
        placebo_treatment = np.random.choice(period_data.index, size=len(period_data)//4, replace=False)
        placebo_treated = period_data.loc[placebo_treatment, 'conversion'].mean()
        placebo_control = period_data.loc[~period_data.index.isin(placebo_treatment), 'conversion'].mean()
        placebo_effect = placebo_treated - placebo_control
        placebo_results.append({'period': period, 'effect': placebo_effect})

print("1. PLACEBO TESTS (should show no significant effects):")
for result in placebo_results:
    print(f"   Period {result['period']}: {result['effect']:8.4f}")

avg_placebo = np.mean([r['effect'] for r in placebo_results])
print(f"   Average placebo effect: {avg_placebo:.4f}")
print(f"   True treatment effect: {treatment_effect:.4f}")

# 2. Sensitivity analysis - exclude different variables
sensitivity_results = {}
for exclude_var in confounders:
    reduced_confounders = [var for var in confounders if var != exclude_var]
    X_reduced = df_historical[reduced_confounders + ['price_experiment']]
    reduced_model = LinearRegression().fit(X_reduced, y)
    reduced_effect = reduced_model.coef_[-1]
    sensitivity_results[exclude_var] = reduced_effect

print(f"\n2. SENSITIVITY ANALYSIS (excluding variables):")
print(f"   Full model effect: {treatment_effect:.4f}")
for var, effect in sensitivity_results.items():
    change = effect - treatment_effect
    print(f"   Excluding {var:15}: {effect:.4f} (change: {change:+.4f})")

# 3. Time-based analysis - check if effects vary over time
time_effects = []
for period in [6, 12, 18]:  # Experiment periods
    period_data = df_historical[df_historical['period'] == period]
    period_treatment = period_data[period_data['price_experiment'] == 1]['conversion'].mean()
    period_control = period_data[period_data['price_experiment'] == 0]['conversion'].mean()
    period_effect = period_treatment - period_control
    time_effects.append({'period': period, 'effect': period_effect})

print(f"\n3. TIME-VARYING EFFECTS:")
for result in time_effects:
    print(f"   Period {result['period']}: {result['effect']:8.4f}")

# 4. Different model specifications
print(f"\n4. DIFFERENT MODEL SPECIFICATIONS:")

# Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X[confounders], y)

# Predict for treatment and control
X_treatment = X[confounders].copy()
X_control = X[confounders].copy()

# Use same confounders but different treatment status
treatment_indices = df_historical[df_historical['price_experiment'] == 1].index
control_indices = df_historical[df_historical['price_experiment'] == 0].index

rf_treatment_pred = rf_model.predict(X.loc[treatment_indices, confounders]).mean()
rf_control_pred = rf_model.predict(X.loc[control_indices, confounders]).mean()
rf_effect = rf_treatment_pred - rf_control_pred

print(f"   Linear regression effect: {treatment_effect:.4f}")
print(f"   Random Forest effect: {rf_effect:.4f}")
print(f"   Difference: {abs(treatment_effect - rf_effect):.4f}")

# Visualize robustness
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
effects = [treatment_effect] + list(sensitivity_results.values())
labels = ['Full Model'] + [f'Exclude {var}' for var in sensitivity_results.keys()]
plt.bar(range(len(effects)), effects, color=['blue'] + ['lightblue'] * (len(effects)-1))
plt.xticks(range(len(effects)), labels, rotation=45, ha='right')
plt.ylabel('Treatment Effect')
plt.title('Sensitivity Analysis')
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
periods = [r['period'] for r in time_effects]
effects = [r['effect'] for r in time_effects]
plt.bar(periods, effects, color=['red', 'green', 'purple'])
plt.xlabel('Experiment Period')
plt.ylabel('Treatment Effect')
plt.title('Time-Varying Effects')
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Business Impact Analysis

Translate causal inference results into business insights.

In [ ]:
print("BUSINESS IMPACT ANALYSIS")
print("=" * 50)

# Calculate business metrics
avg_price_increase = (treatment_group['price'].mean() - control_group['price'].mean()) / control_group['price'].mean()
conversion_effect = treatment_effect
profit_effect = treatment_group['profit'].mean() - control_group['profit'].mean()

print(f"EXPERIMENT RESULTS:")
print(f"Average price increase: {avg_price_increase:.2%}")
print(f"Effect on conversion rate: {conversion_effect:.4f} ({conversion_effect/control_group['conversion'].mean():.2%} relative)")
print(f"Effect on profit per customer: ${profit_effect:.2f}")

# Calculate confidence intervals
treatment_se = treatment_group['conversion'].std() / np.sqrt(len(treatment_group))
control_se = control_group['conversion'].std() / np.sqrt(len(control_group))
effect_se = np.sqrt(treatment_se**2 + control_se**2)

ci_lower = conversion_effect - 1.96 * effect_se
ci_upper = conversion_effect + 1.96 * effect_se

print(f"\n95% Confidence Interval for conversion effect: [{ci_lower:.4f}, {ci_upper:.4f}]")

# Business scenario analysis
monthly_customers = 10000
current_conversion_rate = control_group['conversion'].mean()
current_avg_price = control_group['price'].mean()
current_avg_profit = control_group['profit'].mean()

print(f"\nBUSINESS SCENARIO (for {monthly_customers:,} customers/month):")
print(f"\nCurrent Performance:")
print(f"  Conversion rate: {current_conversion_rate:.2%}")
print(f"  Average price: ${current_avg_price:.2f}")
print(f"  Conversions per month: {monthly_customers * current_conversion_rate:.0f}")
print(f"  Revenue per month: ${monthly_customers * current_conversion_rate * current_avg_price:,.0f}")
print(f"  Profit per month: ${monthly_customers * current_avg_profit:,.0f}")

# With price increase
new_conversion_rate = current_conversion_rate + conversion_effect
new_avg_price = current_avg_price * (1 + avg_price_increase)
new_avg_profit = current_avg_profit + profit_effect

print(f"\nWith Price Increase:")
print(f"  Conversion rate: {new_conversion_rate:.2%}")
print(f"  Average price: ${new_avg_price:.2f}")
print(f"  Conversions per month: {monthly_customers * new_conversion_rate:.0f}")
print(f"  Revenue per month: ${monthly_customers * new_conversion_rate * new_avg_price:,.0f}")
print(f"  Profit per month: ${monthly_customers * new_avg_profit:,.0f}")

# Calculate improvements
revenue_improvement = (monthly_customers * new_conversion_rate * new_avg_price) - (monthly_customers * current_conversion_rate * current_avg_price)
profit_improvement = (monthly_customers * new_avg_profit) - (monthly_customers * current_avg_profit)

print(f"\nIMPROVEMENT POTENTIAL:")
print(f"  Revenue improvement: ${revenue_improvement:,.0f}/month (${revenue_improvement*12:,.0f}/year)")
print(f"  Profit improvement: ${profit_improvement:,.0f}/month (${profit_improvement*12:,.0f}/year)")
print(f"  ROI on price increase: {(profit_improvement / (monthly_customers * current_avg_profit)) * 100:.1f}%")

# Risk assessment
print(f"\nRISK ASSESSMENT:")
if new_conversion_rate < current_conversion_rate:
    conversion_loss = current_conversion_rate - new_conversion_rate
    print(f"  Conversion rate decrease: {conversion_loss:.2%}")
    print(f"  Customers lost per month: {monthly_customers * conversion_loss:.0f}")
else:
    print(f"  Conversion rate maintained or improved")

# Statistical significance
print(f"\nSTATISTICAL SIGNIFICANCE:")
print(f"  Treatment effect p-value: {p_value:.4f}")
print(f"  Statistically significant at α=0.05: {'Yes' if p_value < 0.05 else 'No'}")
print(f"  Effect size (Cohen's d): {simple_diff / np.sqrt(((treatment_group['conversion'].var() + control_group['conversion'].var()) / 2)):.4f}")

print(f"\n" + "="*50)
print("Ready to proceed to P&L Optimization (Notebook 3)")
print("="*50)